# Interaction Lab: standalone cloud qualification and development training

Use a free Colab GPU or Kaggle GPU **interactively in this notebook**. This is a separate development/infrastructure qualification run, not a screening run or the controlled direction experiment. It cannot update either frozen study ledger, choose deployment weights, fit calibration or inspect the locked holdout.

Free GPU allocation and session duration vary. Select a GPU in the platform UI. Upload the exact project source and an already prepared **train + selection-only** cache; do not upload a complete study cache containing calibration/evaluation labels. In this project the selection partition is named `validation` in the cache manifest. No additional dataset is acquired here.

Run the cells in order. Use a unique job ID assigned to this runtime only. A training cell runs at most two hours and then exports a consistent checkpoint snapshot. It is not an SSH service, a distributed worker, a keepalive or a quota workaround. [Colab policies](https://research.google.com/colaboratory/faq.html), [Kaggle usage guidance](https://www.kaggle.com/docs/efficient-gpu-usage).

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys
import time
import uuid

# Colab defaults. On Kaggle use /kaggle/input/<private-input>/adp for REPO_DIR,
# /kaggle/input/<private-input>/cache for SOURCE_CACHE and /kaggle/working/adp-job.
REPO_DIR = Path('/content/adp')
SOURCE_CACHE = Path('/content/input/cache')
WORK_DIR = Path('/content/adp-job')
EXPORT_DIR = Path('/content/exports')  # Or an explicitly mounted private Drive folder.
EXPECTED_MANIFEST_SHA256 = ''  # Paste the SHA-256 from the local train/selection job manifest.
JOB_ID = 'cloud-development-42-v1'  # One owner, never the name of a frozen study run.
TOTAL_EPOCHS = 2
DEVELOPMENT_GPU_HOUR_CAP = 2.0  # Separate qualification budget, includes tests and training.
MAX_SLICE_SECONDS = 2 * 60 * 60

assert sys.version_info[:2] == (3, 12), 'Choose a Python 3.12 runtime; do not change frozen source constraints.'
assert (REPO_DIR / 'pyproject.toml').is_file(), 'Upload the trusted project source and set REPO_DIR.'
assert len(EXPECTED_MANIFEST_SHA256) == 64, 'Set the independently recorded manifest SHA-256 first.'
assert TOTAL_EPOCHS >= 1 and 0 < DEVELOPMENT_GPU_HOUR_CAP <= 2
assert 0 < MAX_SLICE_SECONDS <= 2 * 60 * 60
assert JOB_ID and all(c.isalnum() or c in '-_' for c in JOB_ID)
WORK_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['MKL_NUM_THREADS'] = '2'

Install into this runtime only. Keep a platform-supplied CUDA build if it is PyTorch 2.13.0; do not force a laptop-specific CUDA wheel on an unknown GPU. If the checks fail, stop and qualify an official compatible wheel/runtime separately. No paid tier or account change is required by this notebook.

In [ ]:
import subprocess
import torch

assert str(torch.__version__).split('+')[0] == '2.13.0', 'This job requires a separately qualified PyTorch 2.13.0 runtime.'
assert torch.cuda.is_available(), 'No GPU allocated: check accelerator selection and your current free quota.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)
from adp.training import environment_report
print(json.dumps(environment_report(), indent=2))
print('GPU:', torch.cuda.get_device_name(0), 'capability:', torch.cuda.get_device_capability(0))

Stage only explicitly named training and selection files onto runtime-local storage. The importer validates the manifest hash, IDs and disjointness, rejects calibration/evaluation partitions and symlinks, checks the 8 GiB storage reserve, verifies copied bytes, then publishes the completed cache atomically. The original source cache is preserved.

In [ ]:
import re
import shutil


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def stage_development_cache(source, destination, expected_manifest_sha256):
    source, destination = Path(source).resolve(), Path(destination).resolve()
    if source == destination or source.is_relative_to(destination) or destination.is_relative_to(source):
        raise ValueError('Source and runtime cache must be separate directories.')
    manifest_path = source / 'manifest.json'
    if manifest_path.is_symlink() or sha256_file(manifest_path) != expected_manifest_sha256:
        raise ValueError('Manifest SHA-256 differs from the trusted local job.')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    splits = manifest.get('splits', {})
    if not isinstance(splits, dict) or set(splits) != {'train', 'validation'}:
        raise ValueError('Only train and selection (validation) partitions are accepted.')
    seen, files = set(), [Path('manifest.json'), Path('preprocessing.json')]
    for split in ('train', 'validation'):
        ids = splits[split]
        if not isinstance(ids, list) or not ids:
            raise ValueError('Both train and selection partitions must be nonempty lists.')
        if (source / split).is_symlink():
            raise ValueError('Symlink split directories are not accepted.')
        for scenario_id in ids:
            if not isinstance(scenario_id, str) or not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,127}', scenario_id):
                raise ValueError('Invalid opaque scenario ID.')
            if scenario_id in seen:
                raise ValueError('Scenario IDs must be unique and split-disjoint.')
            seen.add(scenario_id)
            files.append(Path(split) / (scenario_id + '.npz'))
    for relative in files:
        path = source / relative
        if path.is_symlink() or not path.is_file() or not path.resolve().is_relative_to(source):
            raise ValueError('Missing or unsafe cache file: ' + str(relative))
    if destination.exists():
        for relative in files:
            if (destination / relative).is_symlink() or (destination / relative).parent.is_symlink() or not (destination / relative).is_file() or sha256_file(destination / relative) != sha256_file(source / relative):
                raise ValueError('Existing runtime cache differs; choose a fresh WORK_DIR.')
        return destination
    required = sum((source / name).stat().st_size for name in files)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(destination.parent).free - required < 8 * 1024**3:
        raise RuntimeError('Insufficient space: preserve at least 8 GiB after staging.')
    staging = destination.with_name(destination.name + '.staging-' + uuid.uuid4().hex)
    staging.mkdir()
    receipts = {}
    for relative in files:
        target = staging / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(source / relative, target)
        receipts[relative.as_posix()] = sha256_file(target)
        if receipts[relative.as_posix()] != sha256_file(source / relative):
            raise OSError('Copy verification failed; partial staging retained for inspection.')
    (staging / 'import-receipt.json').write_text(json.dumps(receipts, indent=2) + '\n', encoding='utf-8')
    staging.rename(destination)
    return destination

In [ ]:
CACHE_DIR = stage_development_cache(SOURCE_CACHE, WORK_DIR / 'cache', EXPECTED_MANIFEST_SHA256)
RUN_DIR = WORK_DIR / JOB_ID
RUN_DIR.mkdir(exist_ok=True)
source_hash = hashlib.sha256()
source_files = list((REPO_DIR / 'src' / 'adp').rglob('*.py')) + [REPO_DIR/'scripts/qualify-runtime.py', REPO_DIR/'pyproject.toml', REPO_DIR/'config.json']
for path in sorted(source_files):
    source_hash.update(path.relative_to(REPO_DIR).as_posix().encode() + b'\0')
    source_hash.update(bytes.fromhex(sha256_file(path)))
SOURCE_SHA256 = source_hash.hexdigest()
CONFIG_PATH = RUN_DIR / 'config.json'
CONFIG = json.loads((REPO_DIR / 'config.json').read_text())
CONFIG['seed'] = 42
CONFIG['training'].update(batch_size=4, accumulation_steps=8, num_workers=0,
                          checkpoint_seconds=300, minimum_free_gib=8,
                          checkpoint_selection='joint_minFDE')
print('Source SHA-256:', SOURCE_SHA256)
print('Train/selection counts:', {k: len(v) for k, v in json.loads((CACHE_DIR/'manifest.json').read_text())['splits'].items()})

Qualification uses the actual model with the maximum supported 32 actors, 128 map polylines, 50 observed steps, 60 future steps and six shared worlds. The full-shape synthetic batch checks kernels and memory, not prediction quality. FP32 must pass; mixed precision is enabled only if its optimizer step produces finite loss/gradients. The existing portability check also verifies optimizer/RNG continuation. All qualification time is charged to this separate job budget.

In [ ]:
LEDGER_PATH = RUN_DIR / 'development-ledger.json'
ledger = json.loads(LEDGER_PATH.read_text()) if LEDGER_PATH.exists() else {'job_id': JOB_ID, 'epochs': TOTAL_EPOCHS, 'cap_seconds': DEVELOPMENT_GPU_HOUR_CAP * 3600, 'sessions': []}
assert ledger['job_id'] == JOB_ID and ledger['cap_seconds'] == DEVELOPMENT_GPU_HOUR_CAP * 3600 and ledger['epochs'] == TOTAL_EPOCHS
assert all(s['source_sha256'] == SOURCE_SHA256 and s['manifest_sha256'] == EXPECTED_MANIFEST_SHA256 for s in ledger['sessions']), 'Do not resume changed source or a changed dataset under an existing job.'
used_seconds = sum(s.get('elapsed_seconds', s['reserved_seconds']) for s in ledger['sessions'])
remaining_seconds = ledger['cap_seconds'] - used_seconds
assert remaining_seconds > 120, 'Development budget exhausted; do not silently extend it.'
SESSION_STARTED = time.monotonic()
session = {'source_sha256': SOURCE_SHA256, 'manifest_sha256': EXPECTED_MANIFEST_SHA256,
           'reserved_seconds': min(MAX_SLICE_SECONDS, remaining_seconds), 'started_unix': time.time()}
ledger['sessions'].append(session)
LEDGER_PATH.write_text(json.dumps(ledger, indent=2) + '\n')

from adp.data import CachedDataset, collate_scenes
from adp.model import ForecastModel, forecast_loss, model_inputs
from adp.training import _smoke_batch

def full_shape_step(amp):
    device = torch.device('cuda')
    batch = _smoke_batch(device)
    actors = {'history', 'history_mask', 'actor_mask', 'dynamic_mask', 'actor_types', 'future', 'future_mask'}
    maps = {'map_points', 'map_features', 'map_mask'}
    for key, value in list(batch.items()):
        factors = [4] + [1] * (value.ndim - 1)
        if key in actors:
            factors[1] = 16
        elif key in maps:
            factors[1] = 128
        batch[key] = value.repeat(*factors)
    model = ForecastModel(CONFIG['model']).cuda().train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=amp)
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.autocast('cuda', dtype=torch.float16, enabled=amp):
        outputs = model(**model_inputs(batch))
        loss = forecast_loss(outputs, batch)['loss']
    assert outputs['means'].shape == (4, 6, 32, 60, 2)
    assert outputs['scales'].min() >= .15 and outputs['scales'].max() <= 20
    torch.testing.assert_close(outputs['logits'].float().softmax(-1).sum(-1), torch.ones(4, device='cuda'))
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    scaler.step(optimizer)
    scaler.update()
    torch.cuda.synchronize()
    if not amp:
        import io
        model.eval()
        portable = io.BytesIO()
        torch.save({name: value.detach().cpu() for name, value in model.state_dict().items()}, portable)
        portable.seek(0)
        cpu_model = ForecastModel(CONFIG['model']).eval()
        cpu_model.load_state_dict(torch.load(portable, map_location='cpu', weights_only=True))
        cpu_inputs = {key: value[:1].cpu() for key, value in model_inputs(batch).items()}
        with torch.inference_mode():
            expected = cpu_model(**cpu_inputs)
            actual = model(**{key: value.cuda() for key, value in cpu_inputs.items()})
        for key in ('means', 'scales', 'logits', 'risk_logits'):
            torch.testing.assert_close(actual[key].float().cpu(), expected[key].float(), rtol=1e-4, atol=1e-4)
    return {'amp': amp, 'step_seconds': time.perf_counter() - started,
            'peak_allocated_bytes': torch.cuda.max_memory_allocated(), 'loss': float(loss.detach())}

QUALIFIED = False
try:
    qualification = {'environment': environment_report(), 'full_shape_fp32': full_shape_step(False)}
    torch.cuda.empty_cache()
    try:
        qualification['full_shape_amp'] = full_shape_step(True)
        CONFIG['training']['amp'] = True
    except (RuntimeError, AssertionError) as error:
        qualification['amp_failure'] = str(error)
        CONFIG['training']['amp'] = False
    torch.cuda.empty_cache()
    # Existing project check loads only this job's checkpoint, if one exists.
    portability = [sys.executable, str(REPO_DIR/'scripts/qualify-runtime.py'), '--device', 'cuda', '--output', str(RUN_DIR/'portability.json')]
    if (RUN_DIR/'last.pt').is_file():
        portability += ['--checkpoint', str(RUN_DIR/'last.pt')]
    subprocess.run(portability, cwd=REPO_DIR, check=True)
    # Measure a real training batch with labels from train only.
    batch = collate_scenes([CachedDataset(CACHE_DIR, 'train')[i] for i in range(min(4, len(CachedDataset(CACHE_DIR, 'train'))))])
    batch = {k: v.cuda() if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
    model = ForecastModel(CONFIG['model']).cuda().train()
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    torch.cuda.synchronize()
    started = time.perf_counter()
    loss = forecast_loss(model(**model_inputs(batch)), batch)['loss']
    loss.backward()
    assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    optimizer.step()
    torch.cuda.synchronize()
    qualification['real_train_batch'] = {'scenarios': len(batch['scenario_ids']), 'seconds': time.perf_counter() - started,
                                       'loss': float(loss.detach()), 'quality_claim': False}
    del model, optimizer, loss, batch
    torch.cuda.empty_cache()
    if CONFIG_PATH.exists():
        assert json.loads(CONFIG_PATH.read_text()) == CONFIG, 'Qualification changed saved config; use a new job ID.'
    else:
        CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2) + '\n')
    (RUN_DIR/'qualification.json').write_text(json.dumps(qualification, indent=2) + '\n')
    QUALIFIED = True
    print(json.dumps(qualification, indent=2))
except BaseException:
    session['elapsed_seconds'] = time.monotonic() - SESSION_STARTED
    session['status'] = 'qualification_failed'
    LEDGER_PATH.write_text(json.dumps(ledger, indent=2) + '\n')
    raise

Run one bounded training slice. `--epochs` is the fixed total; re-running resumes committed optimizer groups with the same configuration and manifest. Training and selection timing count against this standalone budget. There is no calibration/locked-holdout command. A hard platform shutdown can lose runtime-local changes since the last exported snapshot; free hosting does not guarantee a two-hour session.

In [ ]:
assert QUALIFIED
seconds = min(session['reserved_seconds'] - (time.monotonic() - SESSION_STARTED) - 30, MAX_SLICE_SECONDS)
assert seconds > 30, 'No training time remains after qualification.'
command = [sys.executable, '-m', 'adp', 'train', '--config', str(CONFIG_PATH),
           '--cache-dir', str(CACHE_DIR), '--output-dir', str(RUN_DIR),
           '--epochs', str(TOTAL_EPOCHS), '--device', 'cuda', '--max-runtime-seconds', str(seconds)]
if (RUN_DIR/'last.pt').is_file():
    command += ['--resume', str(RUN_DIR/'last.pt')]
try:
    subprocess.run(command, cwd=REPO_DIR, check=True)
    session['status'] = 'slice_finished'
finally:
    session['elapsed_seconds'] = time.monotonic() - SESSION_STARTED
    LEDGER_PATH.write_text(json.dumps(ledger, indent=2) + '\n')
print('Slice finished. Run the snapshot cell now; active checkpoints remain in place.')

Export only after training has stopped, so each snapshot is consistent. Copy/download the ZIP and SHA-256 receipt to Windows, independently verify the ZIP checksum and all member checksums, then import the checkpoint there. Delete only that acknowledged export snapshot on the cloud side. **Never delete `last.pt`, `best.pt`, the active run directory or source cache** to acknowledge a transfer. Private-server transfers and their two-hour schedule are managed separately by the local transfer tool, not by a free Colab worker.

In [ ]:
import zipfile


def export_snapshot(run_dir, cache_dir, export_dir, source_sha256):
    run_dir, cache_dir, export_dir = Path(run_dir), Path(cache_dir), Path(export_dir)
    checkpoint = run_dir / 'last.pt'
    if not checkpoint.is_file() or checkpoint.is_symlink():
        raise ValueError('No completed resumable checkpoint to export.')
    names = ('last.pt', 'best.pt', 'config.json', 'history.json', 'run.json',
             'training.png', 'qualification.json', 'portability.json', 'development-ledger.json')
    files = [(run_dir/name, name) for name in names if (run_dir/name).is_file()]
    files += [(cache_dir/name, 'cache-provenance/'+name) for name in ('manifest.json', 'preprocessing.json')]
    if any(path.is_symlink() for path, _ in files):
        raise ValueError('Snapshot files must be regular files.')
    export_dir.mkdir(parents=True, exist_ok=True)
    if shutil.disk_usage(export_dir).free - sum(path.stat().st_size for path, _ in files) < 8*1024**3:
        raise RuntimeError('Insufficient space to export while preserving 8 GiB.')
    snapshot_id = time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '-' + uuid.uuid4().hex[:12]
    staging = export_dir / (snapshot_id + '.zip.partial')
    target = export_dir / (snapshot_id + '.zip')
    receipt = {'version': 1, 'snapshot_id': snapshot_id, 'source_sha256': source_sha256,
               'claim': 'development infrastructure qualification; no model promotion',
               'files': {name: {'sha256': sha256_file(path), 'bytes': path.stat().st_size} for path, name in files}}
    with zipfile.ZipFile(staging, 'w', compression=zipfile.ZIP_STORED) as archive:
        for path, name in files:
            archive.write(path, name)
        archive.writestr('snapshot.json', json.dumps(receipt, indent=2)+'\n')
    with zipfile.ZipFile(staging) as archive:
        for name, info in receipt['files'].items():
            if hashlib.sha256(archive.read(name)).hexdigest() != info['sha256']:
                raise OSError('Snapshot changed during export; partial ZIP preserved.')
    staging.rename(target)
    receipt['archive_sha256'] = sha256_file(target)
    receipt_path = target.with_suffix('.receipt.json')
    temporary = receipt_path.with_suffix('.json.partial')
    temporary.write_text(json.dumps(receipt, indent=2)+'\n', encoding='utf-8')
    temporary.rename(receipt_path)
    return target, receipt_path

In [ ]:
snapshot, receipt = export_snapshot(RUN_DIR, CACHE_DIR, EXPORT_DIR, SOURCE_SHA256)
print('Download or copy BOTH files before deleting an export:')
print(snapshot)
print(receipt)
print(receipt.read_text())
# Colab: use the notebook Files pane to download the ZIP and receipt.
# Kaggle: save a private notebook version with outputs, then download both outputs.
# Stop/disconnect the GPU runtime when finished; do not keep idle sessions alive.

## Resume and interpretation

Preserve the source hash, configuration, training-only job manifest, preprocessing metadata, `last.pt` and `best.pt`. Restore only your own verified checkpoint snapshots into the same logical job, and stop the previous owner first. Checkpoint loading is restricted (`weights_only=True`), but still accepts only trusted project-generated artifacts. Requalify each newly allocated GPU.

The per-job ledger conservatively charges a reserved slice if a runtime disappears before recording its actual duration. Do not clear or enlarge it to evade the budget; reconcile it from independent runtime evidence. This notebook does not add hours to the 40-hour screening study or the separate 12-hour direction experiment, and its checkpoints are experimental. Synthetic timing, one real batch and two development epochs establish operability, not accuracy or safety. See `docs/CLOUD_COMPUTE.md` for transfer and platform limits.